# The Price is Right

## Week 8 Order of Play

Day 1: Modal.com and SpecialistAgent  
Day 2: RAG, FrontierAgent, Ensemble Agent  
Day 3: ScannerAgent, MessengerAgent  
Day 4: AutonomousPlannerAgent and DealAgentFramework  
Day 5: The Price Is Right Finale


Today we'll build another piece of the puzzle: a ScanningAgent that looks for promising deals by subscribing to RSS feeds.

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from agents.deals import ScrapedDeal, DealSelection
import logging
import requests
load_dotenv(override=True)
openai = OpenAI()
MODEL = 'gpt-5-mini'

In [2]:
deals = ScrapedDeal.fetch(show_progress=True)

100%|██████████| 3/3 [00:24<00:00,  8.08s/it]


In [3]:
len(deals)

30

In [4]:
deals[10].describe()

'Title: SanDisk Memory Sale at Woot for Deals from $17 + free shipping w/ Prime\nDetails: Woot is selling a bunch of different SanDisk flash drives and memory cards, with 64GB microSD cards starting at $16.99. A 256GB SanDisk Ultra SDXC card is $40, and the 512GB SanDisk Ultra Dual Drive Go USB-C/USB-A drive is $50. The sale also includes Extreme PLUS and Outdoors cards, which are made for action cameras and 4K video. Prime members get free shipping, too. This sale ends November 7. Buy Now at Woot! An Amazon Company\nFeatures: Dual USB-C/USB-A flash drives in 256GB & 512GB capacities SDXC and microSD cards from 64GB to 256GB Extreme PLUS, Ultra PLUS, and Outdoors card lines included Outdoors cards in 4K and FHD versions, some w/ SD adapter Sale ends November 8 at 12AM CT\nURL: https://www.dealnews.com/San-Disk-Memory-Sale-at-Woot-for-Deals-from-17-free-shipping-w-Prime/22330264.html?iref=rss-c39'

### We are going to ask GPT-5-mini to summarize deals and identify their price

In [5]:
SYSTEM_PROMPT = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 
"""

USER_PROMPT_PREFIX = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""

USER_PROMPT_SUFFIX = "\n\nInclude exactly 5 deals, no more."

In [6]:
# this makes a suitable user prompt given scraped deals

def make_user_prompt(scraped):
    user_prompt = USER_PROMPT_PREFIX
    user_prompt += '\n\n'.join([scrape.describe() for scrape in scraped])
    user_prompt += USER_PROMPT_SUFFIX
    return user_prompt

In [7]:
# Let's create a user prompt for the deals we just scraped, and look at how it begins

user_prompt = make_user_prompt(deals)
print(user_prompt[:2000])
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_prompt}]

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

Title: Walmart Resold Flash Deals: Up to 65% off + free shipping w/ $35
Details: This batch of flash deals are ending today in Walmart's Resold section. It includes Apple iPads and iPhones, Samsung smartphones, laptops, and other tech, golf equipment, and home items see big price drops. Free shipping applies over $35; otherwise, it adds $6.99. Pickup may also be available. These resto

In [8]:
response = openai.chat.completions.parse(model=MODEL, messages=messages, response_format=DealSelection, reasoning_effort="minimal")
results = response.choices[0].message.parsed
results

DealSelection(deals=[Deal(product_description='A compact vlogging camera with dual front and rear lenses that can record both scene and on-camera reactions simultaneously. It captures up to 8K video and 75MP photos, includes an 8GB external memory card, a DSLR-style grip with built-in flash, a 2000mAh rechargeable battery, and built-in Wi‑Fi for transferring footage to a phone for editing. The unit is positioned for content creators who want a portable all‑in‑one recording solution with easy phone connectivity.', price=80.0, url='https://www.dealnews.com/Holy-Eye-R7-8-K-Dual-Camera-Vlogging-Camera-for-80-free-shipping/22328150.html?iref=rss-c142'), Deal(product_description='A compact point‑and‑shoot camera with a 13MP CMOS sensor and an f/2.0 26mm fixed wide‑angle lens designed for simple shooting with no zoom or focus adjustments. It has a 2.8" LCD that flips 180° for selfies and vlogging, records 1080p Full HD video, uses a rechargeable built‑in battery with USB‑C charging, and store

In [9]:
for deal in results.deals:
    print(deal.product_description)
    print(deal.price)
    print(deal.url)
    print()


A compact vlogging camera with dual front and rear lenses that can record both scene and on-camera reactions simultaneously. It captures up to 8K video and 75MP photos, includes an 8GB external memory card, a DSLR-style grip with built-in flash, a 2000mAh rechargeable battery, and built-in Wi‑Fi for transferring footage to a phone for editing. The unit is positioned for content creators who want a portable all‑in‑one recording solution with easy phone connectivity.
80.0
https://www.dealnews.com/Holy-Eye-R7-8-K-Dual-Camera-Vlogging-Camera-for-80-free-shipping/22328150.html?iref=rss-c142

A compact point‑and‑shoot camera with a 13MP CMOS sensor and an f/2.0 26mm fixed wide‑angle lens designed for simple shooting with no zoom or focus adjustments. It has a 2.8" LCD that flips 180° for selfies and vlogging, records 1080p Full HD video, uses a rechargeable built‑in battery with USB‑C charging, and stores images on SD cards. The package includes a wrist strap, USB‑C cable, and AC adapter, ma

In [10]:
root = logging.getLogger()
root.setLevel(logging.INFO)

In [11]:
from agents.scanner_agent import ScannerAgent

In [12]:
agent = ScannerAgent()
result = agent.scan()

INFO:root:[Scanner Agent] Scanner Agent is initializing
INFO:root:[Scanner Agent] Scanner Agent is ready
INFO:root:[Scanner Agent] Scanner Agent is about to fetch deals from RSS feed
INFO:root:[Scanner Agent] Scanner Agent received 30 deals not already scraped
INFO:root:[Scanner Agent] Scanner Agent is calling OpenAI using Structured Outputs
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:[Scanner Agent] Scanner Agent received 5 selected deals with price>0 from OpenAI


In [13]:
result

DealSelection(deals=[Deal(product_description='Google Pixel 11 Pro unlocked smartphone with a 6.3-inch 1280×2856 display, 256GB of onboard storage, and a large 4850mAh battery that supports wireless charging. The phone includes advanced imaging capabilities with up to 120x digital zoom, biometric security via fingerprint and facial recognition, and compatibility with major U.S. carriers (unlocked for Google Fi, Verizon, T-Mobile, AT&T, etc.). It supports modern connectivity standards including 5G, Wi‑Fi 6E, and Bluetooth 6.', price=799.0, url='https://www.dealnews.com/Unlocked-Google-Pixel-11-Pro-6-3-256-GB-Smartphone-for-799-free-shipping/22328028.html?iref=rss-c142'), Deal(product_description='AWOL Vision Aetherion Pro ultra short-throw 4K laser home-theater projector delivering 3840×2160 resolution and 2,600 ISO lumens brightness from a close wall distance to fill screens up to 200 inches. It uses an RGB triple-laser light engine, supports Dolby Vision and HDR10+ (also HDR10), offer

### Introducing Pushover

Pushover is a nifty tool for sending Push Notifications to your phone.

It's super easy to set up and install!

Simply visit https://pushover.net/ and click 'Login or Signup' on the top right to sign up for a free account, and create your API keys.

Once you've signed up, on the home screen, click "Create an Application/API Token", and give it any name (like AIEngineer) and click Create Application.

Then add 2 lines to your `.env` file:

PUSHOVER_USER=_put the key that's on the top right of your Pushover home screen and probably starts with a u_  
PUSHOVER_TOKEN=_put the key when you click into your new application called Agents (or whatever) and probably starts with an a_

Remember to save your `.env` file, and run `load_dotenv(override=True)` after saving, to set your environment variables.

Finally, click "Add Phone, Tablet or Desktop" to install on your phone.

In [ ]:
load_dotenv(override=True)

In [ ]:
pushover_user = os.getenv('PUSHOVER_USER')
pushover_token = os.getenv('PUSHOVER_TOKEN')
pushover_url = "https://api.pushover.net/1/messages.json"

In [ ]:
if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

In [ ]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [ ]:
push("MASSIVE DEAL!!")

In [ ]:
from agents.messaging_agent import MessagingAgent

agent = MessagingAgent()
agent.push("SUCH A MASSIVE DEAL!!")

In [ ]:
agent.notify("A special deal on Sumsung 60 inch LED TV going at a great bargain", 300, 1000, "www.samsung.com")